# 05 Multimodal Fusion

No-training multimodal fusion scaffold. Common Voice Yue does not provide emotion labels, so this notebook prepares speech/text/error features and documents the future attention-fusion path without pretending an emotion model exists yet.


## Setup


In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

def find_project_root():
    cwd = Path.cwd().resolve()
    env_root = os.getenv("PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates += [cwd, *cwd.parents]
    candidates += [
        Path(r"D:/GitHub/Cantonese-Speech-AI"),
        Path(r"D:\GitHub\Cantonese-Speech-AI"),
        Path("/mnt/d/GitHub/Cantonese-Speech-AI"),
        Path("/content/Cantonese-Speech-AI"),
        Path("/content/drive/MyDrive/Cantonese-Speech-AI"),
    ]
    for candidate in candidates:
        if (candidate / "src" / "cantonese_speech_ai").exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Cannot find project root from cwd={cwd}")

ROOT = find_project_root()
os.environ["PROJECT_ROOT"] = str(ROOT)
load_dotenv(ROOT / ".env")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

ROOT


WindowsPath('D:/GitHub/Cantonese-Speech-AI')

## Load analysis data


In [2]:
prediction_path = ROOT / "outputs" / "predictions" / "whisper_api_dev_20.csv"
taxonomy_path = ROOT / "outputs" / "analysis" / "tone_error_taxonomy_dev_20.csv"

if not prediction_path.exists():
    prediction_files = sorted(
        (ROOT / "outputs" / "predictions").glob("whisper_api_dev_*.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not prediction_files:
        raise FileNotFoundError("Run 02_asr_baseline.ipynb before this notebook.")
    prediction_path = prediction_files[0]

predictions = pd.read_csv(prediction_path, encoding="utf-8-sig")
taxonomy = pd.read_csv(taxonomy_path, encoding="utf-8-sig") if taxonomy_path.exists() else pd.DataFrame()

analysis = predictions.copy()
if not taxonomy.empty:
    taxonomy_cols = [
        "path",
        "error_category",
        "written_replacement_flags",
        "reference_jyutping",
        "prediction_jyutping",
        "reference_tones",
        "prediction_tones",
        "tone_count_delta",
    ]
    analysis = analysis.merge(taxonomy[taxonomy_cols], on="path", how="left")

analysis["sentence_length"] = analysis["sentence"].fillna("").str.len()
analysis["accents"] = analysis["accents"].fillna("unknown").replace("", "unknown")
analysis["age"] = analysis["age"].fillna("unknown").replace("", "unknown")
analysis["gender"] = analysis["gender"].fillna("unknown").replace("", "unknown")
analysis["error_category"] = analysis["error_category"].fillna("unclassified")
analysis["written_replacement_flags"] = analysis["written_replacement_flags"].fillna("[]")
analysis["has_written_replacement"] = analysis["written_replacement_flags"].ne("[]")
analysis["tone_count_delta"] = analysis["tone_count_delta"].fillna(0)
analysis["high_cer"] = analysis["cer"] >= 0.25

{
    "prediction_path": str(prediction_path),
    "taxonomy_path": str(taxonomy_path),
    "rows": len(analysis),
    "mean_cer": analysis["cer"].mean(),
    "mean_wer": analysis["wer"].mean(),
}


{'prediction_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\predictions\\whisper_api_dev_20.csv',
 'taxonomy_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\analysis\\tone_error_taxonomy_dev_20.csv',
 'rows': 20,
 'mean_cer': np.float64(0.3067809886192239),
 'mean_wer': np.float64(0.8833333333333334)}

## Build multimodal feature scaffold


In [3]:
fusion = analysis.copy()
fusion["reference_text_length"] = fusion["normalized_sentence"].fillna("").str.len()
fusion["prediction_text_length"] = fusion["normalized_prediction"].fillna("").str.len()
fusion["text_length_delta"] = fusion["reference_text_length"] - fusion["prediction_text_length"]
fusion["duration_bucket"] = pd.cut(
    fusion["duration_sec"],
    bins=[0, 3, 5, 8, float("inf")],
    labels=["short", "medium", "long", "very_long"],
)
fusion["pseudo_emotion_label"] = "unlabeled"
fusion["speech_feature_set"] = "duration_only_v1"
fusion["text_feature_set"] = "transcript_prediction_error_features"
fusion["fusion_target"] = "future_emotion_or_intent_label"

feature_columns = [
    "path",
    "duration_sec",
    "duration_bucket",
    "reference_text_length",
    "prediction_text_length",
    "text_length_delta",
    "cer",
    "wer",
    "error_category",
    "has_written_replacement",
    "tone_count_delta",
    "pseudo_emotion_label",
    "speech_feature_set",
    "text_feature_set",
    "fusion_target",
]

fusion_scaffold = fusion[feature_columns]
scaffold_path = ROOT / "outputs" / "analysis" / "multimodal_fusion_scaffold_dev_20.csv"
scaffold_path.parent.mkdir(parents=True, exist_ok=True)
fusion_scaffold.to_csv(scaffold_path, index=False, encoding="utf-8-sig")

{
    "scaffold_path": str(scaffold_path),
    "rows": len(fusion_scaffold),
    "duration_buckets": fusion_scaffold["duration_bucket"].value_counts(dropna=False).to_dict(),
}


{'scaffold_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\analysis\\multimodal_fusion_scaffold_dev_20.csv',
 'rows': 20,
 'duration_buckets': {'medium': 14, 'short': 3, 'long': 3, 'very_long': 0}}

In [4]:
fusion_scaffold.head(20)


,path,duration_sec,duration_bucket,reference_text_length,prediction_text_length,text_length_delta,cer,wer,error_category,has_written_replacement,tone_count_delta,pseudo_emotion_label,speech_feature_set,text_feature_set,fusion_target
0,common_voice_yue_39013334.mp3,5.292,long,15,15,0,0.066667,0.500000,low_error,False,0,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label
1,common_voice_yue_39013050.mp3,3.348,medium,7,7,0,0.000000,0.000000,low_error,False,0,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label
2,common_voice_yue_39013032.mp3,3.492,medium,6,6,0,0.833333,1.000000,high_cer_written_style,True,0,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label
3,common_voice_yue_39013090.mp3,6.660,long,21,21,0,0.333333,1.000000,high_cer_written_style,True,0,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label
4,common_voice_yue_39013110.mp3,4.896,medium,15,14,1,0.600000,1.000000,high_cer_written_style,True,1,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label
5,common_voice_yue_39013113.mp3,3.204,medium,8,7,1,0.250000,1.000000,high_cer,False,1,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label
6,common_voice_yue_39013129.mp3,3.060,medium,6,6,0,0.333333,1.000000,high_cer,False,0,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label
7,common_voice_yue_39013130.mp3,2.628,short,5,5,0,0.000000,0.000000,low_error,False,0,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label
8,common_voice_yue_39013034.mp3,3.456,medium,8,8,0,0.250000,1.000000,high_cer_written_style,True,0,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label
9,common_voice_yue_39013286.mp3,3.276,medium,8,8,0,0.625000,1.000000,high_cer_written_style,True,0,unlabeled,duration_only_v1,transcript_prediction_error_features,future_emotion_or_intent_label


## Future fusion architecture


In [5]:
fusion_architecture = {
    "speech_encoder": {
        "v1_features": ["duration_sec", "duration_bucket"],
        "future_features": ["F0_contour", "energy", "pause_rate", "speaker_embedding", "emotion_embedding"],
    },
    "text_encoder": {
        "v1_features": ["normalized_sentence", "normalized_prediction", "error_category"],
        "future_features": ["semantic_embedding", "colloquial_marker_embedding", "tone_embedding"],
    },
    "fusion_module": {
        "recommended": "gated_cross_attention",
        "alternatives": ["late_fusion_mlp", "concat_projection", "adversarial_domain_alignment"],
    },
    "training_status": "blocked_until_emotion_or_intent_labels_are_available",
}
fusion_architecture


{'speech_encoder': {'v1_features': ['duration_sec', 'duration_bucket'],
  'future_features': ['F0_contour',
   'energy',
   'pause_rate',
   'speaker_embedding',
   'emotion_embedding']},
 'text_encoder': {'v1_features': ['normalized_sentence',
   'normalized_prediction',
   'error_category'],
  'future_features': ['semantic_embedding',
   'colloquial_marker_embedding',
   'tone_embedding']},
 'fusion_module': {'recommended': 'gated_cross_attention',
  'alternatives': ['late_fusion_mlp',
   'concat_projection',
   'adversarial_domain_alignment']},
 'training_status': 'blocked_until_emotion_or_intent_labels_are_available'}

## Weak supervision note

Before training multimodal fusion, add either a labeled emotion dataset or a carefully reviewed pseudo-label source. For this Common Voice-only phase, use this notebook as a feature and architecture scaffold.
